In [1]:
"""
load_eGFR_models.py  (v4-2 기준)
═══════════════════════════════════════════════════════════════════════════════
다른 인스턴스에서 저장된 eGFR25 모델 패키지를 로딩하는 독립 스크립트.

사용법
──────
    python load_eGFR25_models.py --pkg_dir /home/ubuntu/eGFR_output/models
    python load_eGFR2_models.py --pkg_dir ./models --cpu   # GPU 없는 환경
"""

import argparse, json, sys
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from tabpfn.model_loading import load_fitted_tabpfn_model


def load_package(pkg_dir: str | Path, map_to_cpu: bool = False) -> dict:
    """
    저장된 모델 패키지 전체를 로딩합니다.

    Returns
    -------
    {
      "models_A"  : dict[model_name → Pipeline],
      "models_B"  : dict[model_name → Pipeline],
      "imputer_A" : TwoTierImputer,
      "imputer_B" : TwoTierImputer,
      "metadata"  : dict,
    }
    """
    pkg = Path(pkg_dir)

    # ── Step 1. TwoTierImputer 클래스 등록 (joblib.load 전 필수) ─────────────
    src_dir = str(pkg / "src")
    if src_dir not in sys.path:
        sys.path.insert(0, src_dir)
    try:
        import two_tier_imputer as _tti
        import __main__
        __main__.TwoTierImputer = _tti.TwoTierImputer
        print(f"[LOAD] TwoTierImputer: {_tti.__file__} ✅")
    except ImportError as e:
        raise ImportError(
            f"two_tier_imputer.py를 찾을 수 없습니다.\n"
            f"  탐색 경로: {src_dir}\n"
            f"  해결: {src_dir} 에 two_tier_imputer.py가 있는지 확인하세요.\n"
            f"  오류: {e}"
        ) from e

    # ── Step 2. metadata ──────────────────────────────────────────────────────
    with open(pkg / "metadata.json", encoding="utf-8") as f:
        metadata = json.load(f)
    model_order = metadata["model_order"]
    model_specs = metadata["model_specs"]
    print(f"[LOAD] metadata: {len(model_order)} models | "
          f"horizon={metadata.get('horizon_days')} days")
    if "key_package_versions" in metadata:
        print(f"[LOAD] 저장 당시 패키지: "
              f"{list(metadata['key_package_versions'].values())}")

    # ── Step 3. 모델 로딩 ─────────────────────────────────────────────────────
    result = {"models_A": {}, "models_B": {}, "metadata": metadata}

    for scen_tag, key in [("scenario_A", "models_A"), ("scenario_B", "models_B")]:
        scen_dir = pkg / scen_tag
        if not scen_dir.exists():
            print(f"[LOAD] ⚠️  {scen_tag} 폴더 없음 — 건너뜁니다")
            continue

        for name in model_order:
            pre_path = scen_dir / "preprocess" / f"{name}_preprocess.joblib"
            tpf_path = scen_dir / "tabpfn_fit"  / f"{name}.tabpfn_fit"
            if not pre_path.exists() or not tpf_path.exists():
                print(f"[LOAD] ⚠️  {scen_tag}/{name} 파일 없음 — 건너뜁니다")
                continue

            pre = joblib.load(pre_path)
            clf = load_fitted_tabpfn_model(tpf_path)
            if map_to_cpu and hasattr(clf, "to"):
                clf = clf.to("cpu")

            result[key][name] = Pipeline(steps=[("preprocess", pre), ("clf", clf)])
            print(f"[LOAD] ✅ {scen_tag}/{name}")

        imp_path = scen_dir / "imputer.joblib"
        if imp_path.exists():
            imp     = joblib.load(imp_path)
            imp_key = "imputer_A" if scen_tag == "scenario_A" else "imputer_B"
            result[imp_key] = imp
            print(f"[LOAD] ✅ {scen_tag}/imputer  (include_cap={imp.include_cap})")

    return result


def predict(
    pkg: dict,
    df_new: pd.DataFrame,
    scenario: str = "B",
    model_name: str = "base_lsm_cap",
) -> np.ndarray:
    """
    로딩된 패키지로 새 데이터 예측 확률을 반환합니다.

    전처리 순서 (자동 처리):
        1. add_fibroscan_date_numeric()  ← Scenario B only
        2. imputer.transform()
        3. pipeline.predict_proba()

    Returns
    -------
    np.ndarray, shape (n_samples,)  — CKD 발생 예측 확률
    """
    from two_tier_imputer import add_fibroscan_date_numeric

    imp      = pkg[f"imputer_{scenario}"]
    pipe     = pkg[f"models_{scenario}"][model_name]
    features = pkg["metadata"]["model_specs"][model_name]

    df_prep = df_new.copy()
    if scenario == "B" and "Fibroscan_Date_numeric" not in df_prep.columns:
        df_prep = add_fibroscan_date_numeric(df_prep)

    df_imp = imp.transform(df_prep)
    return pipe.predict_proba(df_imp[features])[:, 1]


if __name__ == "__main__":
    parser = argparse.ArgumentParser(description="CKD 모델 패키지 로더")
    parser.add_argument("--pkg_dir", required=True, help="models/ 디렉터리 경로")
    parser.add_argument("--cpu", action="store_true", help="GPU→CPU 리매핑")
    sys.argv = ['', '--pkg_dir', '/home/ubuntu/eGFR25_output/models']
    args = parser.parse_args()

    pkg = load_package(args.pkg_dir, map_to_cpu=args.cpu)

    print("\n[SUMMARY] 로딩 완료")
    for scen in ["A", "B"]:
        key = f"models_{scen}"
        if key in pkg and pkg[key]:
            print(f"  Scenario {scen}: {list(pkg[key].keys())}")

[LOAD] TwoTierImputer: /home/ubuntu/eGFR25_output/models/src/two_tier_imputer.py ✅
[LOAD] metadata: 6 models | horizon=1826 days
[LOAD] 저장 당시 패키지: ['numpy==2.3.5', 'pandas==2.3.3', 'scikit-learn==1.8.0', 'tabpfn==6.4.1', 'tabpfn-common-utils==0.2.16', 'torch==2.7.1+cu118', 'torchaudio==2.7.1+cu118', 'torchvision==0.22.1+cu118']
[LOAD] ✅ scenario_A/base
[LOAD] ✅ scenario_A/base_lsm
[LOAD] ✅ scenario_A/base_lsm_cap
[LOAD] ✅ scenario_A/base_inflam
[LOAD] ✅ scenario_A/base_inflam_lsm
[LOAD] ✅ scenario_A/base_inflam_lsm_cap
[LOAD] ✅ scenario_A/imputer  (include_cap=False)
[LOAD] ✅ scenario_B/base
[LOAD] ✅ scenario_B/base_lsm
[LOAD] ✅ scenario_B/base_lsm_cap
[LOAD] ✅ scenario_B/base_inflam
[LOAD] ✅ scenario_B/base_inflam_lsm
[LOAD] ✅ scenario_B/base_inflam_lsm_cap
[LOAD] ✅ scenario_B/imputer  (include_cap=True)

[SUMMARY] 로딩 완료
  Scenario A: ['base', 'base_lsm', 'base_lsm_cap', 'base_inflam', 'base_inflam_lsm', 'base_inflam_lsm_cap']
  Scenario B: ['base', 'base_lsm', 'base_lsm_cap', 'base_i

In [2]:
EPS = 1e-12
def _clip(p): return np.clip(p, EPS, 1-EPS)
def _logit(p): p=_clip(p); return np.log(p/(1-p))

def bh_fdr(pvals):
    pvals=np.asarray(pvals,dtype=float); n=len(pvals)
    if n==0: return pvals
    order=np.argsort(pvals)
    q=pvals[order]*n/(np.arange(n)+1)
    q=np.minimum.accumulate(q[::-1])[::-1]
    out=np.empty_like(q); out[order]=np.clip(q,0,1)
    return out

def stratified_bootstrap_indices(y, rng, n_boot=500):
    y=np.asarray(y).astype(int)
    i0,i1=np.where(y==0)[0],np.where(y==1)[0]
    for _ in range(n_boot):
        idx=np.concatenate([rng.choice(i0,len(i0),replace=True),
                             rng.choice(i1,len(i1),replace=True)])
        rng.shuffle(idx); yield idx

def ci_from_boot(d,alpha=0.05):
    d=np.asarray(d,dtype=float)
    if d.size==0: return np.nan,np.nan
    return float(np.quantile(d,alpha/2)),float(np.quantile(d,1-alpha/2))

def two_sided_p(d):
    d=np.asarray(d,dtype=float)
    if d.size==0: return np.nan
    return float(min(1.0,2*min(np.mean(d<=0),np.mean(d>=0))))

def disc_metrics(y,p):
    y=np.asarray(y).astype(int); p=np.asarray(p,dtype=float)
    if len(np.unique(y))<2: return {"AUROC":np.nan,"AUPRC":np.nan}
    return {"AUROC":float(roc_auc_score(y,p)),
            "AUPRC":float(average_precision_score(y,p))}

def boot_delta_disc(y,pa,pb,n_boot=500,seed=42):
    y=np.asarray(y).astype(int); pa=np.asarray(pa,dtype=float); pb=np.asarray(pb,dtype=float)
    rng=np.random.default_rng(seed); da,dp=[],[]
    for idx in stratified_bootstrap_indices(y,rng,n_boot):
        yb=y[idx]
        if len(np.unique(yb))<2: continue
        da.append(roc_auc_score(yb,pb[idx])-roc_auc_score(yb,pa[idx]))
        dp.append(average_precision_score(yb,pb[idx])-average_precision_score(yb,pa[idx]))
    return np.asarray(da,dtype=float),np.asarray(dp,dtype=float)

def cal_metrics(y,p):
    y=np.asarray(y).astype(int); p=np.asarray(p,dtype=float)
    lr=LogisticRegression(penalty=None,solver="lbfgs",max_iter=2000)
    lr.fit(_logit(p).reshape(-1,1),y)
    return {"Brier":float(brier_score_loss(y,p)),
            "Cal_Intercept":float(lr.intercept_[0]),
            "Cal_Slope":float(lr.coef_[0][0])}

def nri_cat(y,po,pn,cp=(0.05,0.10,0.20)):
    y=np.asarray(y).astype(int); bins=np.array([0.,*cp,1.+1e-9])
    co=np.digitize(np.asarray(po,dtype=float),bins)-1
    cn=np.digitize(np.asarray(pn,dtype=float),bins)-1
    up,dn=cn>co,cn<co; ev,ne=y==1,y==0
    return float((np.mean(up[ev])-np.mean(dn[ev]))+(np.mean(dn[ne])-np.mean(up[ne])))

def nri_cont(y,po,pn):
    y=np.asarray(y).astype(int)
    d=np.asarray(pn,dtype=float)-np.asarray(po,dtype=float)
    up,dn=d>0,d<0; ev,ne=y==1,y==0
    return float((np.mean(up[ev])-np.mean(dn[ev]))+(np.mean(dn[ne])-np.mean(up[ne])))

def boot_nri(y,po,pn,n_boot=500,seed=42,cp=(0.05,0.10,0.20)):
    y=np.asarray(y).astype(int)
    po=np.asarray(po,dtype=float); pn=np.asarray(pn,dtype=float)
    rng=np.random.default_rng(seed); dc,dk=[],[]
    for idx in stratified_bootstrap_indices(y,rng,n_boot):
        dc.append(nri_cat(y[idx],po[idx],pn[idx],cp))
        dk.append(nri_cont(y[idx],po[idx],pn[idx]))
    return np.asarray(dc,dtype=float),np.asarray(dk,dtype=float)

def harrell_c(t,e,r):
    t=np.asarray(t,dtype=float); e=np.asarray(e,dtype=int); r=np.asarray(r,dtype=float)
    conc=perm=ties=0.0; n=len(t)
    for i in range(n):
        for j in range(i+1,n):
            if e[i]==1 and t[i]<t[j]:
                perm+=1
                if r[i]>r[j]: conc+=1
                elif r[i]==r[j]: ties+=1
            elif e[j]==1 and t[j]<t[i]:
                perm+=1
                if r[j]>r[i]: conc+=1
                elif r[i]==r[j]: ties+=1
    return np.nan if perm==0 else float((conc+0.5*ties)/perm)

def boot_cindex(t,e,r,n_boot=700,seed=42):
    rng=np.random.default_rng(seed)
    dist=[harrell_c(t[idx],e[idx],r[idx])
          for idx in stratified_bootstrap_indices(np.asarray(e).astype(int),rng,n_boot)]
    dist=[x for x in dist if np.isfinite(x)]
    lo,hi=ci_from_boot(np.asarray(dist))
    return harrell_c(t,e,r),lo,hi

def boot_delta_cindex(t,e,pa,pb,n_boot=500,seed=42):
    rng=np.random.default_rng(seed); dist=[]
    for idx in stratified_bootstrap_indices(np.asarray(e).astype(int),rng,n_boot):
        ca=harrell_c(t[idx],e[idx],pa[idx]); cb=harrell_c(t[idx],e[idx],pb[idx])
        if np.isfinite(ca) and np.isfinite(cb): dist.append(cb-ca)
    return np.asarray(dist,dtype=float)

def build_surv(df,event_col,tte_col,fu_col,horizon):
    ev=pd.to_numeric(df[event_col],errors="coerce").values
    tte=pd.to_numeric(df[tte_col],errors="coerce").values
    fu=pd.to_numeric(df[fu_col],errors="coerce").values
    t=np.minimum(fu,horizon).astype(float)
    e=np.zeros(len(df),dtype=int)
    m=(ev==1)&(tte>0)&(tte<=horizon)&np.isfinite(tte)
    e[m]=1; t[m]=tte[m]
    return t,e


def evaluate_models_full(
    y_true, pred_proba, df_split,
    event_col, tte_col, fu_col="Follow-up_duration",
    horizon_days=1826, n_boot=500, seed=42
):
    names=list(pred_proba.keys())
    print(f"[EVAL] models={names}")

    # per-model
    summary=pd.DataFrame(
        [{"Model":m, **disc_metrics(y_true,pred_proba[m]),
          **cal_metrics(y_true,pred_proba[m])} for m in names])

    # pairwise disc
    pr,pauc,ppr=[],[],[]
    for i in range(len(names)):
        for j in range(i+1,len(names)):
            a,b=names[i],names[j]
            da,dp=boot_delta_disc(y_true,pred_proba[a],pred_proba[b],n_boot,seed)
            la,ha=ci_from_boot(da); lp,hp=ci_from_boot(dp)
            pa_v,pp_v=two_sided_p(da),two_sided_p(dp)
            pauc.append(pa_v); ppr.append(pp_v)
            pr.append({"Compare":f"{b} - {a}",
                       "Delta_AUROC":float(np.nanmean(da)) if da.size else np.nan,
                       "Delta_AUROC_CI_low":la,"Delta_AUROC_CI_high":ha,
                       "p_AUROC":float(pa_v) if np.isfinite(pa_v) else np.nan,
                       "Delta_AUPRC":float(np.nanmean(dp)) if dp.size else np.nan,
                       "Delta_AUPRC_CI_low":lp,"Delta_AUPRC_CI_high":hp,
                       "p_AUPRC":float(pp_v) if np.isfinite(pp_v) else np.nan})
    pairwise=pd.DataFrame(pr)
    if len(pairwise):
        pairwise["q_AUROC_BH"]=bh_fdr(np.array(pauc,dtype=float))
        pairwise["q_AUPRC_BH"]=bh_fdr(np.array(ppr,dtype=float))

    # NRI — pre-specified comparisons
    # ─────────────────────────────────────────────────────────────────────────
    # Groups and their scientific questions:
    #
    # Within-B1  (Block 1, no inflammation adjustment)
    #   base → base_lsm          : incremental value of LSM (fibrosis marker)
    #   base_lsm → base_lsm_cap  : incremental value of CAP (steatosis) on top of fibrosis
    #   base → base_lsm_cap      : combined FibroScan phenotype value
    #
    # Within-B2  (Block 2, inflammation-adjusted with AST/ALT)
    #   Same 3 comparisons as B1, but after conditioning on inflammatory state.
    #   Key: does LSM / CAP retain incremental value once inflammation is controlled?
    #
    # AST-ALT-standalone
    #   base → base_inflam        : independent predictive value of AST+ALT alone
    #   (required to validate that Block 2 base model is meaningfully better
    #    than Block 1 base before comparing further)
    #
    # Cross-block  (same FibroScan model, B1 vs B2)
    #   base_lsm     vs base_inflam_lsm    : does LSM effect attenuate with AST/ALT adjustment?
    #   base_lsm_cap vs base_inflam_lsm_cap: does combined FibroScan effect attenuate?
    # ─────────────────────────────────────────────────────────────────────────
    nri_pairs=[]  # (reference, new, group_label)

    # Within Block 1
    if "base"     in pred_proba and "base_lsm"     in pred_proba: nri_pairs.append(("base",    "base_lsm",     "Within-B1"))
    if "base_lsm" in pred_proba and "base_lsm_cap" in pred_proba: nri_pairs.append(("base_lsm","base_lsm_cap", "Within-B1"))
    if "base"     in pred_proba and "base_lsm_cap" in pred_proba: nri_pairs.append(("base",    "base_lsm_cap", "Within-B1"))
    # AST/ALT standalone value (prerequisite check for Block 2 interpretation)
    if "base" in pred_proba and "base_inflam" in pred_proba: nri_pairs.append(("base", "base_inflam", "AST-ALT-standalone"))

    # Within Block 2
    if "base_inflam"     in pred_proba and "base_inflam_lsm"     in pred_proba: nri_pairs.append(("base_inflam",    "base_inflam_lsm",     "Within-B2"))
    if "base_inflam_lsm" in pred_proba and "base_inflam_lsm_cap" in pred_proba: nri_pairs.append(("base_inflam_lsm","base_inflam_lsm_cap", "Within-B2"))
    if "base_inflam"     in pred_proba and "base_inflam_lsm_cap" in pred_proba: nri_pairs.append(("base_inflam",    "base_inflam_lsm_cap", "Within-B2"))
    # Cross-block: attenuation of FibroScan markers after inflammation adjustment
    if "base_lsm"     in pred_proba and "base_inflam_lsm"     in pred_proba: nri_pairs.append(("base_lsm",    "base_inflam_lsm",     "Cross-block"))
    if "base_lsm_cap" in pred_proba and "base_inflam_lsm_cap" in pred_proba: nri_pairs.append(("base_lsm_cap","base_inflam_lsm_cap", "Cross-block"))

    nr,pc_all,pk_all=[],[],[]
    for ref,new,grp in nri_pairs:
        dc,dk=boot_nri(y_true,pred_proba[ref],pred_proba[new],n_boot,seed)
        lc,hc=ci_from_boot(dc); lk,hk=ci_from_boot(dk)
        pc,pk=two_sided_p(dc),two_sided_p(dk)
        pc_all.append(pc); pk_all.append(pk)
        nr.append({"Group":grp, "Reference":ref, "Compare":f"{new} - {ref}",
                   "CatNRI":float(np.nanmean(dc)) if dc.size else np.nan,
                   "CatNRI_CI_low":lc,"CatNRI_CI_high":hc,
                   "p_CatNRI":float(pc) if np.isfinite(pc) else np.nan,
                   "ContNRI":float(np.nanmean(dk)) if dk.size else np.nan,
                   "ContNRI_CI_low":lk,"ContNRI_CI_high":hk,
                   "p_ContNRI":float(pk) if np.isfinite(pk) else np.nan})
    nri_tbl=None
    if nr:
        nri_tbl=pd.DataFrame(nr)
        nri_tbl["q_CatNRI_BH"]=bh_fdr(np.array(pc_all,dtype=float))
        nri_tbl["q_ContNRI_BH"]=bh_fdr(np.array(pk_all,dtype=float))

    # C-index
    st,se=build_surv(df_split,event_col,tte_col,fu_col,horizon_days)
    ci_rows=[]
    for k,m in enumerate(names):
        cp,lo,hi=boot_cindex(st,se,pred_proba[m],n_boot,seed+100+k)
        ci_rows.append({"Model":m,"Harrell_C":cp,"CI_low":lo,"CI_high":hi})
    cindex=pd.DataFrame(ci_rows)
    cindex["Harrell_C_95CI"]=cindex.apply(
        lambda r:f'{r.Harrell_C:.3f} ({r.CI_low:.3f}\u2013{r.CI_high:.3f})',axis=1)

    # pairwise C-index
    cp_rows,pc2=[],[]
    for i in range(len(names)):
        for j in range(i+1,len(names)):
            a,b=names[i],names[j]
            d=boot_delta_cindex(st,se,pred_proba[a],pred_proba[b],n_boot,seed)
            lo,hi=ci_from_boot(d); p=two_sided_p(d); pc2.append(p)
            cp_rows.append({"Compare":f"{b} - {a}",
                            "Delta_Cindex":float(np.nanmean(d)) if d.size else np.nan,
                            "Delta_Cindex_CI_low":lo,"Delta_Cindex_CI_high":hi,
                            "p_Cindex":float(p) if np.isfinite(p) else np.nan})
    pairwise_c=pd.DataFrame(cp_rows)
    if len(pairwise_c):
        pairwise_c["q_Cindex_BH"]=bh_fdr(np.array(pc2,dtype=float))

    print("[EVAL] Done.")
    return dict(summary=summary, pairwise_discrimination=pairwise,
                nri=nri_tbl, cindex=cindex, pairwise_cindex=pairwise_c)

print("[CELL 9] Evaluation functions defined.")

[CELL 9] Evaluation functions defined.


In [3]:
import time
from pathlib import Path

OUTDIR = Path("/home/ubuntu/eGFR25_output")
OUTDIR.mkdir(parents=True, exist_ok=True)

def run_eval(label, y, pred, df_imp, event_col, tte_col, n_boot, seed):
    print()
    print("="*80)
    print(f"  {label}")
    print("="*80)
    t0 = time.time()
    result = evaluate_models_full(
        y, pred, df_imp,
        event_col=event_col, tte_col=tte_col,
        n_boot=n_boot, seed=seed
    )
    elapsed = time.time() - t0
    mins, secs = divmod(int(elapsed), 60)
    print(f"  ✅ Done in {mins}m {secs}s")
    return result

total_t0 = time.time()

res_A_test = run_eval("Scenario A — Internal test (CAP-era)",
    y_test_A, pred_test_A, df_test_A_imp,
    event_col=event_col, tte_col=tte_col, n_boot=N_BOOT, seed=RANDOM_STATE)

res_A_ext = run_eval("Scenario A — External validation (CAP-era)",
    y_ext_A, pred_ext_A, df_ext_A_imp,
    event_col=event_col, tte_col=tte_col, n_boot=N_BOOT, seed=RANDOM_STATE)

res_B_test = run_eval("Scenario B — Internal test (full+MICE+aux)",
    y_test, pred_test_B, df_test_B_imp,
    event_col=event_col, tte_col=tte_col, n_boot=N_BOOT, seed=RANDOM_STATE)

res_B_ext = run_eval("Scenario B — External validation (full+MICE+aux)",
    y_ext, pred_ext_B, df_ext_B_imp,
    event_col=event_col, tte_col=tte_col, n_boot=N_BOOT, seed=RANDOM_STATE)

total_elapsed = time.time() - total_t0
total_mins, total_secs = divmod(int(total_elapsed), 60)
print()
print("="*80)
print(f"  🎉 All evaluations complete — total time: {total_mins}m {total_secs}s")
print("="*80)

NameError: name 'y_test_A' is not defined

In [4]:
# 스크립트 2에 정의된 load_package 함수 실행
pkg = load_package("/home/ubuntu/eGFR25_output/models")

[LOAD] TwoTierImputer: /home/ubuntu/eGFR25_output/models/src/two_tier_imputer.py ✅
[LOAD] metadata: 6 models | horizon=1826 days
[LOAD] 저장 당시 패키지: ['numpy==2.3.5', 'pandas==2.3.3', 'scikit-learn==1.8.0', 'tabpfn==6.4.1', 'tabpfn-common-utils==0.2.16', 'torch==2.7.1+cu118', 'torchaudio==2.7.1+cu118', 'torchvision==0.22.1+cu118']
[LOAD] ✅ scenario_A/base
[LOAD] ✅ scenario_A/base_lsm
[LOAD] ✅ scenario_A/base_lsm_cap
[LOAD] ✅ scenario_A/base_inflam
[LOAD] ✅ scenario_A/base_inflam_lsm
[LOAD] ✅ scenario_A/base_inflam_lsm_cap
[LOAD] ✅ scenario_A/imputer  (include_cap=False)
[LOAD] ✅ scenario_B/base
[LOAD] ✅ scenario_B/base_lsm
[LOAD] ✅ scenario_B/base_lsm_cap
[LOAD] ✅ scenario_B/base_inflam
[LOAD] ✅ scenario_B/base_inflam_lsm
[LOAD] ✅ scenario_B/base_inflam_lsm_cap
[LOAD] ✅ scenario_B/imputer  (include_cap=True)


In [11]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

# ── 1. 원본 데이터 로드 (본인의 실제 경로로 수정하세요) ───────────────────────
# 1번 스크립트의 '1. Configuration' 섹션에서 사용했던 파일 경로입니다.
DATA_PATH = "/home/ubuntu/Fibroscan_CKD_2025.xlsx"
def load_df_raw(path: str) -> pd.DataFrame:
    if not os.path.exists(path):
        raise FileNotFoundError(f"[LOAD] File not found: {path}")
    ext = os.path.splitext(path)[1].lower()
    if ext == ".csv":
        df = pd.read_csv(path)
    elif ext in [".xlsx", ".xls"]:
        df = pd.read_excel(path)
    else:
        raise ValueError(f"[LOAD] Unsupported extension: {ext}")
    print(f"[LOAD] df_raw loaded: shape={df.shape}, columns={len(df.columns)}")
    return df

df_raw = load_df_raw(DATA_PATH)

def apply_cohort_exclusions(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    for col in ["Baseline_eGFR","ESRD","KT","Baseline_PU","Malignancy","Follow-up_duration"]:
        if col in out.columns:
            out[col] = pd.to_numeric(out[col], errors="coerce")
    print(f"[EXCLUDE] Start n={len(out):,}")
    mask = pd.Series(True, index=out.index)
    for col, crit, label in [
        ("Baseline_eGFR",       lambda d: d < 60, "Baseline CKD (eGFR<60)"),
        ("ESRD",                lambda d: d == 1, "ESRD==1"),
        ("KT",                  lambda d: d == 1, "KT==1"),
        ("Baseline_PU",         lambda d: d == 1, "Baseline_PU==1"),
        ("Malignancy",          lambda d: d == 1, "Malignancy==1"),
        ("Follow-up_duration",  lambda d: d < MIN_FU_DAYS_FOR_COHORT, "FU<1Y"),
    ]:
        if col in out.columns:
            n_before = mask.sum()
            mask &= ~crit(out[col])
            print(f"[EXCLUDE] {label}: removed {n_before - mask.sum():,}")
    out = out.loc[mask].reset_index(drop=True)
    print(f"[EXCLUDE] End n={len(out):,}")
    return out


def build_5yr_outcome(
    df: pd.DataFrame, event_col: str, tte_col: str,
    fu_col: str = "Follow-up_duration", five_year_days: int = FIVE_YEARS_DAYS,
) -> pd.DataFrame:
    out = df.copy()
    for col in [event_col, tte_col, fu_col]:
        out[col] = pd.to_numeric(out[col], errors="coerce")
    print(f"[OUTCOME] Building y for {event_col} within 5y")
    n0   = len(out)
    cand = out.loc[out[fu_col].notna() & ((out[event_col]!=1) | out[tte_col].notna())].copy()
    print(f"[OUTCOME] After drop missing FU/TTE: {len(cand):,} (removed {n0-len(cand):,})")
    y = pd.Series(np.nan, index=cand.index)
    y.loc[(cand[event_col]==1)&(cand[tte_col]>0)&(cand[tte_col]<=five_year_days)] = 1
    y.loc[(cand[event_col]==0)&(cand[fu_col]>=five_year_days)]                    = 0
    y.loc[(cand[event_col]==1)&(cand[tte_col]>five_year_days)&(cand[fu_col]>=five_year_days)] = 0
    before = len(cand)
    cand   = cand.loc[y.notna()].copy()
    cand["y"] = y.loc[cand.index].astype(int)
    print(f"[OUTCOME] Removed uncertain: {before-len(cand):,}")
    print(f"[OUTCOME] Final n={len(cand):,}, event rate={cand['y'].mean():.3f}")
    return cand.reset_index(drop=True)

OUTCOME = "eGFR25"   # 'CKD' or 'eGFR25'
event_col, tte_col = ("eGFR25","TTO_eGFR25") if OUTCOME=="eGFR25" else ("eGFR25","TTO_eGFR25")

df_excl  = apply_cohort_exclusions(df_raw)
df_final = build_5yr_outcome(df_excl, event_col=event_col, tte_col=tte_col)

print("\n[FINAL COHORT] by Hospital")
print(df_final.groupby("Hospital")["y"].agg(N="count", Events="sum", EventRate="mean"))

# ── 2. 1번 스크립트와 동일한 상수로 데이터 분할 ──────────────────────────────
HOSP_INTERNAL = 10  
HOSP_EXTERNAL = 20    
RANDOM_STATE = 42
MAX_TABPFN_N = 10000

internal = df_final[df_final["Hospital"] == HOSP_INTERNAL].copy()
external = df_final[df_final["Hospital"] == HOSP_EXTERNAL].copy()

# 1번 스크립트와 동일한 인덱스 분할 (random_state=42가 핵심)
idx_train, idx_test = train_test_split(
    internal.index.values, test_size=0.2, 
    random_state=RANDOM_STATE, stratify=internal["y"].values
)

df_test = internal.loc[idx_test].copy()
df_ext  = external.copy()

# ── 3. Scenario A/B용 변수 및 레이블 생성 ─────────────────────────────────────
event_col = 'y'

# Scenario A: CAP 데이터가 있는 샘플만 추출
df_test_A = df_test.dropna(subset=['CAP']).copy()
df_ext_A  = df_ext.dropna(subset=['CAP']).copy()

y_test_A = df_test_A[event_col].astype(int).values
y_ext_A  = df_ext_A[event_col].astype(int).values
y_test   = df_test[event_col].astype(int).values
y_ext    = df_ext[event_col].astype(int).values

# ── 4. 저장된 모델로 예측값(Probability) 재생성 ───────────────────────────────
# 스크립트 2 상단에서 이미 로드된 'pkg'를 사용합니다.
MODEL_ORDER = pkg["metadata"]["model_order"]

print("재생성 시작 (학습이 아니므로 금방 끝납니다)...")
pred_test_A = {m: predict(pkg, df_test_A, scenario='A', model_name=m) for m in MODEL_ORDER}
pred_ext_A  = {m: predict(pkg, df_ext_A,  scenario='A', model_name=m) for m in MODEL_ORDER}
pred_test_B = {m: predict(pkg, df_test,   scenario='B', model_name=m) for m in MODEL_ORDER}
pred_ext_B  = {m: predict(pkg, df_ext,    scenario='B', model_name=m) for m in MODEL_ORDER}

# 평가 함수(run_eval)에 전달할 데이터프레임 매칭
df_test_A_imp = df_test_A
df_ext_A_imp  = df_ext_A
df_test_B_imp = df_test
df_ext_B_imp  = df_ext

print(f"[OK] 데이터 복구 완료: Test_A n={len(y_test_A)}, Test_B n={len(y_test)}")

[LOAD] df_raw loaded: shape=(59161, 60), columns=60


NameError: name 'FIVE_YEARS_DAYS' is not defined